# EDA: Bike Sharing

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), "../.."))

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from ucimlrepo import fetch_ucirepo

from src.preprocess import add_cyclic_hour

## Load Dataset

In [ ]:
bike_sharing = fetch_ucirepo(id=275)

X = bike_sharing.data.features.copy()
y = bike_sharing.data.targets.copy()

print(f"Features : {X.shape}")
print(f"Targets  : {y.shape}")

## Initial Exploration

In [ ]:
X.head()

In [ ]:
X.info()

In [ ]:
y.head()

| Column       | Role     | Type        | Description                                              |
|--------------|----------|-------------|----------------------------------------------------------|
| instant      | ID       | Integer     | Record index                                             |
| dteday       | Feature  | Date        | Date                                                     |
| season       | Feature  | Categorical | 1=Winter, 2=Spring, 3=Summer, 4=Fall                     |
| yr           | Feature  | Categorical | 0=2011, 1=2012                                           |
| mnth         | Feature  | Categorical | Month (1–12)                                             |
| hr           | Feature  | Categorical | Hour (0–23)                                              |
| holiday      | Feature  | Binary      | Whether the day is a holiday                             |
| weekday      | Feature  | Categorical | Day of the week                                          |
| workingday   | Feature  | Binary      | 1 if neither weekend nor holiday                         |
| weathersit   | Feature  | Categorical | 1=Clear … 4=Heavy Rain                                   |
| temp         | Feature  | Continuous  | Normalised temperature in Celsius (−8 to 39)             |
| atemp        | Feature  | Continuous  | Normalised feeling temperature (−16 to 50)               |
| hum          | Feature  | Continuous  | Normalised humidity ÷ 100                                |
| windspeed    | Feature  | Continuous  | Normalised wind speed ÷ 67                               |
| casual       | Other    | Integer     | Count of casual users                                    |
| registered   | Other    | Integer     | Count of registered users                                |
| cnt          | Target   | Integer     | Total rentals (casual + registered)                      |


### Missing Values

In [ ]:
missing = X.isna().sum()
print(missing[missing > 0] if missing.any() else "No missing values in features.")
print()
print("Target missing:", y.isna().sum().item())

## Univariate Analysis

### Numerical Features

In [ ]:
num_features = ["temp", "atemp", "hum", "windspeed"]


def univariate_num(df, feature):
    summary = df[feature].describe()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=df, x=feature, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {feature}")
    sns.boxplot(x=df[feature], ax=axes[1])
    axes[1].set_title(f"Boxplot of {feature}")
    plt.tight_layout()
    plt.show()
    return summary


for feat in num_features:
    print(f"── {feat} ──")
    print(univariate_num(X, feat))
    print()

#### Target: `cnt`

In [ ]:
univariate_num(y, "cnt")

### Categorical Features

In [ ]:
cat_features = [
    "season",
    "yr",
    "mnth",
    "hr",
    "holiday",
    "weekday",
    "workingday",
    "weathersit",
]


def univariate_cat(df, feature):
    counts = df[feature].value_counts()
    percentages = df[feature].value_counts(normalize=True) * 100
    summary = pd.DataFrame({"Count": counts, "Percentage (%)": percentages})
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=feature, order=counts.index)
    plt.title(f"Distribution of {feature}")
    plt.tight_layout()
    plt.show()
    return summary


for feat in cat_features:
    print(f"── {feat} ──")
    print(univariate_cat(X, feat))
    print()

## Bivariate Analysis

### Numerical Features × Target

In [ ]:
def bivariate_num(X, y, feature):
    data = pd.concat([X[[feature]], y], axis=1)
    data.columns = [feature, "cnt"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.scatterplot(data=data, x=feature, y="cnt", alpha=0.3, ax=axes[0])
    axes[0].set_title(f"{feature} vs cnt")
    sns.regplot(data=data, x=feature, y="cnt", scatter_kws={"alpha": 0.2}, ax=axes[1])
    axes[1].set_title(f"Regression: {feature} vs cnt")
    plt.tight_layout()
    plt.show()


for feat in num_features:
    print(f"── {feat} ──")
    bivariate_num(X, y, feat)

### Categorical Features × Target

In [ ]:
def bivariate_cat(X, y, feature):
    data = pd.concat([X[[feature]], y], axis=1)
    data.columns = [feature, "cnt"]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.barplot(data=data, x=feature, y="cnt", ax=axes[0])
    axes[0].set_title(f"Mean cnt by {feature}")
    sns.boxplot(data=data, x=feature, y="cnt", ax=axes[1])
    axes[1].set_title(f"Boxplot cnt by {feature}")
    sns.violinplot(data=data, x=feature, y="cnt", ax=axes[2])
    axes[2].set_title(f"Violin cnt by {feature}")
    plt.tight_layout()
    plt.show()


for feat in cat_features:
    print(f"── {feat} ──")
    bivariate_cat(X, y, feat)

### Pairplot (Numerical Features + Target)

In [ ]:
sns.pairplot(pd.concat([X[num_features], y], axis=1))
plt.show()

### Correlation Heatmap

In [ ]:
corr = pd.concat([X[num_features], y], axis=1).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, cmap="coolwarm", annot=True, fmt=".2f")
plt.title("Pearson Correlation Matrix")
plt.tight_layout()
plt.show()

**Observations:**
- `temp` and `atemp` are almost perfectly linearly correlated (r ≈ 0.99). Since `atemp` has higher mutual information with the target, we will **drop `temp`**.
- `hum` and `windspeed` have a slight negative relationship with `cnt`.


## Preprocessing

### Outlier / Rare Category: `weathersit == 4`

In [ ]:
print("weathersit value counts:")
print(X["weathersit"].value_counts())
print()
# Only 3 records with weathersit=4 (Heavy Rain/Ice) — remove them
mask = X["weathersit"] != 4
X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)
print(f"Dataset after removing weathersit=4: {X.shape}")

### Feature Selection via Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_regression

X_mi = X.drop(columns=["dteday"]).copy()
discrete_mask = X_mi.columns.isin(cat_features)

mi = mutual_info_regression(
    X_mi, y.values.ravel(), discrete_features=discrete_mask, random_state=42
)
mi_df = pd.DataFrame({"Feature": X_mi.columns, "MI": mi}).sort_values(
    "MI", ascending=False
)

plt.figure(figsize=(9, 5))
sns.barplot(data=mi_df, x="MI", y="Feature")
plt.title("Mutual Information with Target (cnt)")
plt.tight_layout()
plt.show()

mi_df

### Feature Engineering Decisions

| Decision | Reason |
|---|---|
| Drop `instant`, `dteday` | ID / date string — no predictive value after other time features are included |
| Drop `casual`, `registered` | Data leakage: they sum to `cnt` |
| Drop `temp` | Perfectly collinear with `atemp` (r = 0.99); `atemp` is more informative |
| Drop `holiday`, `weekday` | Low MI; `workingday` already captures the working/non-working distinction |
| Drop `mnth` | Redundant with `season`; much higher cardinality for marginal gain |
| Cyclical encode `hr` | Hours are periodic — sin/cos avoids a false ordinal gap between 23 and 0 |
| One-hot encode `season`, `weathersit` | Nominal categoricals with no natural order |


### Correlation between `mnth` and `season`

In [ ]:
tab = pd.crosstab(X["mnth"], X["season"])
chi2, p, dof, _ = chi2_contingency(tab)
n = tab.sum().sum()
cramers_v = np.sqrt((chi2 / n) / (min(tab.shape) - 1))
print(f"Cramér's V (mnth × season): {cramers_v:.3f}  (p={p:.2e})")
print("=> Strong association — drop mnth, keep season.")

### Final Feature Set Preview

`add_cyclic_hour` from `src.preprocess` takes a column (array-like) and returns `(sin_hour, cos_hour)`. Below we apply it to preview the final feature set without modifying the working copies of `X` and `y`.


In [ ]:
# Preview only — does not modify X / y used above
X_preview = X.drop(columns=["dteday", "temp", "holiday", "weekday", "mnth"]).copy()

X_preview["hr_sin"], X_preview["hr_cos"] = add_cyclic_hour(X_preview["hr"])
X_preview.drop(columns=["hr"], inplace=True)

X_preview = pd.get_dummies(X_preview, columns=["season", "weathersit"], drop_first=True)
bool_cols = X_preview.select_dtypes("bool").columns
X_preview[bool_cols] = X_preview[bool_cols].astype(int)

print(f"Final feature count: {X_preview.shape[1]}")
print(X_preview.dtypes)